# Test Set Evaluation

Loads all 15 checkpoints (5 models x 3 seeds) from Drive, runs inference on the held-out test split, and aggregates mean +/- std per model across seeds.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'

import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
!unzip -q "$DATASET_ZIP" -d /content/data

In [ ]:
import pandas as pd
import torch

from evaluate import (
    load_single_task_model, load_multitask_model,
    predict_single_task, predict_multitask,
    count_params, measure_inference_time_ms,
)
from metrics import classification_metrics, evaluate_multitask

test_df = pd.read_csv('/content/repo/02_Manifests/test.csv')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 43, 44]
len(test_df), DEVICE

In [ ]:
single_task_results = []
single_task_configs = [
    {'name': 'ModelA_species', 'task': 'species', 'num_classes': 8},
    {'name': 'ModelB_freshness', 'task': 'freshness', 'num_classes': 3},
]

for cfg in single_task_configs:
    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        ckpt_path = f'{CHECKPOINT_DIR}/{run_name}.pt'
        model = load_single_task_model(ckpt_path, cfg['num_classes'], DEVICE)
        y_true, y_pred = predict_single_task(model, test_df, DATASET_ROOT, cfg['task'], DEVICE)
        metrics = classification_metrics(y_true, y_pred, ordinal=(cfg['task'] == 'freshness'))
        metrics.update({
            'model': cfg['name'], 'seed': seed, 'task': cfg['task'],
            'params': count_params(model),
            'inference_ms': measure_inference_time_ms(model, DEVICE),
        })
        single_task_results.append(metrics)

single_task_df = pd.DataFrame(single_task_results)
single_task_df

In [ ]:
multitask_results = []
multitask_names = ['ModelC_EW', 'ModelC_UW', 'ModelC_DWA']

for name in multitask_names:
    for seed in SEEDS:
        run_name = f'{name}_seed{seed}'
        ckpt_path = f'{CHECKPOINT_DIR}/{run_name}.pt'
        model = load_multitask_model(ckpt_path, DEVICE)
        sp_t, sp_p, fr_t, fr_p = predict_multitask(model, test_df, DATASET_ROOT, DEVICE)
        report = evaluate_multitask(sp_t, sp_p, fr_t, fr_p)
        multitask_results.append({
            'model': name, 'seed': seed,
            'species_accuracy': report['species']['accuracy'],
            'species_precision_macro': report['species']['precision_macro'],
            'species_recall_macro': report['species']['recall_macro'],
            'species_f1_macro': report['species']['f1_macro'],
            'species_mcc': report['species']['mcc'],
            'species_cohen_kappa': report['species']['cohen_kappa'],
            'freshness_accuracy': report['freshness']['accuracy'],
            'freshness_precision_macro': report['freshness']['precision_macro'],
            'freshness_recall_macro': report['freshness']['recall_macro'],
            'freshness_f1_macro': report['freshness']['f1_macro'],
            'freshness_mcc': report['freshness']['mcc'],
            'freshness_qwk': report['freshness']['qwk'],
            'joint_accuracy': report['joint_accuracy'],
            'params': count_params(model),
            'inference_ms': measure_inference_time_ms(model, DEVICE),
        })

multitask_df = pd.DataFrame(multitask_results)
multitask_df

## Aggregate across seeds

In [ ]:
multitask_df.groupby('model').agg(['mean', 'std']).round(4)

In [ ]:
single_task_df.groupby(['model', 'task'])[['accuracy', 'f1_macro', 'mcc', 'params', 'inference_ms']].agg(['mean', 'std']).round(4)

## Confusion matrices for one representative run

Set `INSPECT_RUN_NAME` to the run to inspect (e.g. the best-performing seed of the best model, once known from the aggregation above).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from metrics import task_confusion_matrix

INSPECT_RUN_NAME = 'ModelC_DWA_seed42'  # set from the ranking above

model = load_multitask_model(f'{CHECKPOINT_DIR}/{INSPECT_RUN_NAME}.pt', DEVICE)
sp_t, sp_p, fr_t, fr_p = predict_multitask(model, test_df, DATASET_ROOT, DEVICE)

species_labels = sorted(test_df['species'].unique())
freshness_labels = ['Highly Fresh', 'Fresh', 'Not Fresh']

species_cm = task_confusion_matrix(sp_t, sp_p)
freshness_cm = task_confusion_matrix(fr_t, fr_p)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(species_cm, annot=True, fmt='d', xticklabels=species_labels, yticklabels=species_labels, ax=axes[0], cmap='Blues')
axes[0].set_title('Species'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
sns.heatmap(freshness_cm, annot=True, fmt='d', xticklabels=freshness_labels, yticklabels=freshness_labels, ax=axes[1], cmap='Blues')
axes[1].set_title('Freshness'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, f'{INSPECT_RUN_NAME}_confusion_matrices.png'), dpi=150)
plt.show()

In [ ]:
single_task_df.to_csv(os.path.join(RESULTS_DIR, 'single_task_test_results.csv'), index=False)
multitask_df.to_csv(os.path.join(RESULTS_DIR, 'multitask_test_results.csv'), index=False)
print('saved to', RESULTS_DIR)

Results are saved to `RESULTS_DIR` on Drive, not to `/content/repo` -- that clone is deleted when this runtime recycles. To add these results to the GitHub repository, download the two CSV files from Drive and commit them from a machine with push access.